# Train a model

Models are plain `nn.Module`s that take packed tensors, so there is no framework to learn: a standard PyTorch training loop works as-is. This notebook writes that loop out in full, step by step, and trains PointNet++ on ModelNet40. The last section shows the same run through the :lightning: [PyTorch Lightning](https://lightning.ai/) wrappers in `torch_pointcloud.lightning`, which trade the explicit loop for less code.

The dataset downloads on first run. `LIMIT_BATCHES` below keeps the default run to a handful of batches so every cell finishes in seconds; set it to `None` for a real run.

In [ ]:
import torch
from torch.utils.data import Subset

import torch_pointcloud as tp
import torch_pointcloud.transforms as T
from torch_pointcloud.utils.data import PointCloudDataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_POINTS = 1024
BATCH_SIZE = 32
EPOCHS = 1
LIMIT_BATCHES = 4

torch.manual_seed(0)
DEVICE

## Data

ModelNet40 ships CAD meshes, so the pipeline samples points on the faces and normalizes the result. The train pipeline adds augmentation on top; the eval pipeline stays deterministic so validation numbers are comparable across epochs.

In [ ]:
from torch_pointcloud.datasets import ModelNet40

sample = T.Compose([
    T.Rescale(keys="pos", method="centroid"),
    T.RandomSampleFaceVertices(keys="pos", face_key="face", num_samples=NUM_POINTS),
    T.KeepItems(keys=["pos", "normal", "label"]),
])
train_tf = T.Compose([
    sample,
    T.RandomScale(keys="pos", scale_range=(0.8, 1.2)),
    T.RandomJitter(keys="pos", sigma=0.01, clip=0.05),
])

train_dataset = ModelNet40(root="data", train=True, transform=train_tf, download=True)
val_dataset = ModelNet40(root="data", train=False, transform=sample, download=True)
len(train_dataset), len(val_dataset)

### Packed batches

Point clouds have different point counts, so batches are *packed* rather than padded: every cloud is concatenated along axis 0 and a `batch` index tags each point with the cloud it came from. `PointCloudDataLoader` is a `DataLoader` with that collation wired in.

`KeepItems` above matters here: without it every batch would also carry the raw mesh faces the points were sampled from, which are far larger than the sample itself.

In [ ]:
if LIMIT_BATCHES is not None:
    n = LIMIT_BATCHES * BATCH_SIZE
    train_dataset = Subset(train_dataset, torch.randperm(len(train_dataset))[:n].tolist())
    val_dataset = Subset(val_dataset, torch.randperm(len(val_dataset))[:n].tolist())

train_loader = PointCloudDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = PointCloudDataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

data = next(iter(train_loader))
{k: tuple(v.shape) for k, v in data.items() if torch.is_tensor(v)}

`pos` is $(N, 3)$ with $N = 32 \times 1024$ points from 32 clouds, `batch` is $(N,)$, and `label` is $(B,)$: one class per cloud.

## Model

`create_model` builds the architecture. `pretrained=False` (the default) gives random weights, which is what training from scratch wants.

In [ ]:
model = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification").to(DEVICE)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")
print("signature: model(x, pos, batch)")
print("in_channels:", model.in_channels, "| num_classes:", model.num_classes)

Classification models take `(x, pos, batch)` and return $(B, C)$ logits. This configuration has `in_channels = 0`, meaning it learns from geometry alone, so `x` is `None`. A model configured with `in_channels = 6` would instead receive `x = torch.cat([pos, normal], dim=1)`.

## Optimizer, scheduler, loss

Nothing point-cloud-specific here: these are the stock `torch.optim` objects.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(train_loader))
criterion = torch.nn.CrossEntropyLoss()

## The training loop

One epoch of training is the familiar five steps, with the only library-specific part being how a batch dict is unpacked onto the device:

1. move `pos`, `batch` and `label` to the device
2. `optimizer.zero_grad()`
3. forward, then loss
4. `loss.backward()`
5. `optimizer.step()`

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, total_correct, total_seen = 0.0, 0, 0

    for data in loader:
        pos = data["pos"].to(device)
        batch = data["batch"].to(device)
        target = data["label"].to(device)

        optimizer.zero_grad()
        logits = model(None, pos, batch)
        loss = criterion(logits, target)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * target.numel()
        total_correct += int(logits.argmax(dim=1).eq(target).sum())
        total_seen += target.numel()

    return {"loss": total_loss / total_seen, "acc": total_correct / total_seen}

Evaluation is the same walk over the data without the backward pass, under `torch.no_grad()` and with the model in `eval()` mode.

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0

    for data in loader:
        pos = data["pos"].to(device)
        batch = data["batch"].to(device)
        target = data["label"].to(device)

        logits = model(None, pos, batch)
        total_loss += criterion(logits, target).item() * target.numel()
        total_correct += int(logits.argmax(dim=1).eq(target).sum())
        total_seen += target.numel()

    return {"loss": total_loss / total_seen, "acc": total_correct / total_seen}

Driving both from an epoch loop is all that is left. With `LIMIT_BATCHES` set, the numbers below come from a few batches, so they only show that the loop runs. The next section runs the same loop over the whole dataset.

In [ ]:
for epoch in range(EPOCHS):
    train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE)
    val_metrics = evaluate(model, val_loader, criterion, DEVICE)
    print(
        f"epoch {epoch + 1}/{EPOCHS}"
        f" | train loss {train_metrics['loss']:.3f} acc {train_metrics['acc']:.3f}"
        f" | val loss {val_metrics['loss']:.3f} acc {val_metrics['acc']:.3f}"
    )

### A full run

The loop above is written out for teaching. The repo ships the same architecture as a Hydra experiment, and the figures below come from that run:

```text
uv run --no-sync python train.py experiment=pointnet2/modelnet40 logger=tensorboard run_name=modelnet40_tutorial
```

`configs/experiment/pointnet2/modelnet40.yaml` holds the recipe, and writes checkpoints, a config snapshot and TensorBoard events under `logs/train/runs/modelnet40_tutorial/<timestamp>/`.

Point TensorBoard at the run directory:

```bash
uv run --no-sync tensorboard --logdir logs/train/runs/modelnet40_tutorial
```

![The TensorBoard scalars pane: train/loss, val/accuracy and val/loss over 60k steps.](../assets/tutorials/tensorboard_scalars.png)

### Where the errors are

Overall accuracy says nothing about which of the 40 classes go wrong. A confusion matrix does: reload the run's best checkpoint, replay the test split, and count how often each true class lands on each predicted one.

`return_info=True` hands back the evaluation transform the checkpoint was validated with, so the replay preprocesses its input the way the run did.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassConfusionMatrix

from torch_pointcloud.datasets import ModelNetNormalResampled
from torch_pointcloud.lightning import LitClassificationModel

RUN_DIR = sorted(Path("logs/train/runs/modelnet40_tutorial").glob("*"))[-1]
checkpoint = sorted(RUN_DIR.glob("checkpoints/epoch_*.ckpt"))[-1]
trained = LitClassificationModel.load_from_checkpoint(checkpoint, map_location=DEVICE).model
trained.eval().to(DEVICE)

_, info = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification", return_info=True)
test_dataset = ModelNetNormalResampled(root="data", variant="40", train=False, transform=info["transform"])
test_loader = PointCloudDataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

metric = MulticlassConfusionMatrix(num_classes=40)
with torch.no_grad():
    for data in test_loader:
        logits = trained(None, data["pos"].to(DEVICE), data["batch"].to(DEVICE))
        metric.update(logits.argmax(dim=1).cpu(), data["label"])

confusion = metric.compute()
rows = confusion / confusion.sum(dim=1, keepdim=True).clamp(min=1)

_, ax = plt.subplots(figsize=(11.0, 10.0))
image = ax.imshow(rows, cmap="Oranges", vmin=0.0, vmax=1.0)
ax.set_xticks(range(40), test_dataset.classes, rotation=90, fontsize=8)
ax.set_yticks(range(40), test_dataset.classes, fontsize=8)
ax.set(xlabel="predicted", ylabel="true")
plt.colorbar(image, ax=ax, label="fraction of a true class")
plt.show()

![A 40 by 40 confusion matrix of the ModelNet40 run, with a solid diagonal and a few bright off-diagonal cells.](../assets/tutorials/training_confusion.png)

Every row is one true class and sums to 1, so the diagonal is that class's recall and the bright cells beside it are where its meshes went instead. The bright off-diagonal cells fall on the near-duplicate categories: ModelNet40 keeps `plant` and `flower_pot`, `desk` and `table`, `dresser` and `night_stand` as separate classes, and 1024 sampled points do not always tell them apart.

In [ ]:
recall = confusion.diag() / confusion.sum(dim=1).clamp(min=1)
ranked = sorted(zip(test_dataset.classes, recall.tolist()), key=lambda entry: entry[1])

print(f"overall {confusion.diag().sum() / confusion.sum():.3f} | mean per class {recall.mean():.3f}")
for name, value in ranked[:5]:
    print(f"{name:<12} {value:.2f}")

_, ax = plt.subplots(figsize=(7.0, 9.0))
ax.barh([name for name, _ in ranked], [value for _, value in ranked], height=0.7)
ax.axvline(recall.mean(), linewidth=0.9, linestyle="--")
ax.invert_yaxis()  # hardest class at the top
ax.set(xlabel="recall", xlim=(0, 1))
plt.show()

![One horizontal bar per ModelNet40 class, sorted with the hardest class first, with the mean marked by a dashed line.](../assets/tutorials/training_per_class.png)

The diagonal of the same matrix is the per-class accuracy, and its mean is the second number ModelNet40 results usually quote. It weighs a class with 20 test meshes as heavily as one with 100, and 18 of the 40 classes have only 20, so it sits below the overall figure whenever the small classes are the hard ones. The classes at the top of the sorted bars are the ones to look up in the matrix above.

## Save and reload

Weights are a plain `state_dict`. Reload them into a fresh architecture built by the same `create_model` call.

In [ ]:
torch.save(model.state_dict(), "pointnet2_modelnet40.pt")

reloaded = tp.create_model("pointnet2-ssg.modelnet40.xu-yan", task="classification")
reloaded.load_state_dict(torch.load("pointnet2_modelnet40.pt", weights_only=True))
reloaded.eval();

## The same run with Lightning

Everything above is the loop Lightning would otherwise write for you. The `lightning` extra (`pip install "torch-pointcloud[lightning]"`) provides two wrappers:

- `PointCloudDataModule` builds the packed loaders from the datasets.
- `LitClassificationModel` takes the same model name `create_model` does, and owns the loop, the logging and the checkpointing.

In [ ]:
from functools import partial

import lightning.pytorch as L
from torchmetrics.classification import MulticlassAccuracy

from torch_pointcloud.lightning import LitClassificationModel, MetricCallback, PointCloudDataModule

dm = PointCloudDataModule(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    batch_size=BATCH_SIZE,
    num_workers=0,
)
lit = LitClassificationModel(
    "pointnet2-ssg.modelnet40.xu-yan",
    optimizer=partial(torch.optim.AdamW, lr=1e-3, weight_decay=1e-4),
    scheduler=partial(torch.optim.lr_scheduler.CosineAnnealingLR, T_max=200),
    target_key="label",
)

trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=False,
    enable_checkpointing=False,
    callbacks=[MetricCallback(MulticlassAccuracy(num_classes=40), name="acc")],
)
trainer.fit(lit, datamodule=dm)

`trainer.validate` replays the validation loop on its own, printing the same `val/loss` and `val/acc` the plain loop computed by hand.

In [ ]:
trainer.validate(lit, datamodule=dm)

Give the `Trainer` a logger instead of `logger=False` and it writes the same event files the full run above wrote:

```python
from lightning.pytorch.loggers import TensorBoardLogger

trainer = L.Trainer(max_epochs=EPOCHS, logger=TensorBoardLogger("logs", name="my_run"))
```

`read_scalars("logs/my_run/version_0", ["train/loss", "val/loss"])` then reads those scalars back, which is how every curve on this page was drawn.

## Beyond this notebook

- **Segmentation.** The loop is identical except the target is per-point: read `data["segment"]` and pass `ignore_index` to the loss. With Lightning, use `LitSegmentationModel` with `target_key="segment"` and optionally `mix_prob` for Mix3D.
- **Longer epochs.** Wrap the train dataset in `RepeatDataset(dataset, loop=k)` to take more optimizer steps per epoch.
- **Layer-wise learning rates.** Build the parameter groups with `torch_pointcloud.utils.optim.generate_param_groups`, or pass `param_groups=...` to the LightningModule.
- **Full recipes.** The :github: [`examples/`](https://github.com/arthurdjn/pytorch-pointcloud/tree/main/examples) directory has end-to-end training and benchmark scripts for most architectures.